### source/deep-learning-from-scratch-2/ch07/generate_text.py

In [2]:
# coding: utf-8
import sys
sys.path.append('./source/deep-learning-from-scratch-2')
sys.path.append('./source/deep-learning-from-scratch-2/ch07')
from rnnlm_gen import RnnlmGen
from data import ptb


corpus, word_to_id, id_to_word = ptb.load_data('train')
vocab_size = len(word_to_id)
corpus_size = len(corpus)

model = RnnlmGen()
model.load_params('./source/deep-learning-from-scratch-2/ch06/Rnnlm.pkl')

# start 문자와 skip 문자 설정
start_word = 'you'
start_id = word_to_id[start_word]
skip_words = ['N', '<unk>', '$']
skip_ids = [word_to_id[w] for w in skip_words]
# 문장 생성
word_ids = model.generate(start_id, skip_ids)
txt = ' '.join([id_to_word[i] for i in word_ids])
txt = txt.replace(' <eos>', '.\n')
print(txt)

you foes who minivans with clients they could prevent the invitation of the government 's decision to join a buy authority.
 decide we 're not going to make only it a strong view of goldsmith 's junk fund you want to plan for restrictions.
 otherwise prospect index advanced to the stock and debt in the wake of a broad compromise area freely too as mr. lawson 's decision in his business that had been involved.
 we shake possibly two-thirds rather than one mr. azoff said.
 the commissioner of where mr. baldwin is not going to the


## source/deep-learning-from-scratch-2/ch07/generate_better_text.py

In [7]:
# coding: utf-8
import sys
sys.path.append('./source/deep-learning-from-scratch-2')
sys.path.append('./source/deep-learning-from-scratch-2/ch07')
from common.np import *
from rnnlm_gen import RnnlmGen  # BetterRnnlmGen 대신 RnnlmGen 사용
from data import ptb


corpus, word_to_id, id_to_word = ptb.load_data('train')
vocab_size = len(word_to_id)
corpus_size = len(corpus)


model = RnnlmGen()  # BetterRnnlmGen 대신 RnnlmGen 사용
model.load_params('./source/deep-learning-from-scratch-2/ch06/Rnnlm.pkl')

# start 문자와 skip 문자 설정
start_word = 'you'
start_id = word_to_id[start_word]
skip_words = ['N', '<unk>', '$']
skip_ids = [word_to_id[w] for w in skip_words]
# 문장 생성
word_ids = model.generate(start_id, skip_ids)
txt = ' '.join([id_to_word[i] for i in word_ids])
txt = txt.replace(' <eos>', '.\n')

print(txt)


model.reset_state()

start_words = 'the meaning of life is'
start_ids = [word_to_id[w] for w in start_words.split(' ')]

for x in start_ids[:-1]:
    x = np.array(x).reshape(1, 1)
    model.predict(x)

word_ids = model.generate(start_ids[-1], skip_ids)
word_ids = start_ids[:-1] + word_ids
txt = ' '.join([id_to_word[i] for i in word_ids])
txt = txt.replace(' <eos>', '.\n')
print('-' * 50)
print(txt)

you think they 're impossible to serve when measures are n't going to be a idea of animals.
 nor could help the possibility of much much.
 however three years later at least specialized issues mr. roman also said according to international business assistance to the premium settlement by suit are divided into hopes of john.
 mr. roman is a executive vice president of the separate league where supreme court shall help salvage criminal agent readers.
 he also said dioxide intended for a partial proposal from the soviet name is seeking to take the securities since next
--------------------------------------------------
the meaning of life is based for columbia.
 the prohibits the rothschilds break with plans for its move for saab airlines and three other manufacturers.
 but it created a third of his high-tech language and limits on how too soon he offering to new york for anacomp equity human services and rubicam for all powerful bankruptcy-law.
 the plan is angry and encouraged among the

In [ ]:
def train_cpu_optimized_model(train_papers, val_papers):
    """CPU 최적화 모델 훈련"""
    print("🚀 CPU 최적화 Adaptive Attention T5 훈련 시작")
    
    try:
        # 토크나이저 로드
        tokenizer = T5Tokenizer.from_pretrained(CONFIG["model_name"], legacy=False)
        config = T5Config.from_pretrained(CONFIG["model_name"])
        model = LightweightT5WithAdaptiveAttention(config)
        model = model.to(device)
        
        print(f"🔧 모델 파라미터: {sum(p.numel() for p in model.parameters()):,}")
        
        # Dataset 생성
        train_dataset = ArxivDataset(train_papers, tokenizer, 
                                   CONFIG["max_input_length"], CONFIG["max_target_length"])
        val_dataset = ArxivDataset(val_papers, tokenizer,
                                 CONFIG["max_input_length"], CONFIG["max_target_length"])
        
        # CPU 최적화 훈련 설정
        training_args = TrainingArguments(
            output_dir=CONFIG["output_dir"],
            num_train_epochs=CONFIG["num_epochs"],
            per_device_train_batch_size=CONFIG["batch_size"],
            per_device_eval_batch_size=CONFIG["batch_size"],
            learning_rate=CONFIG["learning_rate"],
            warmup_steps=100,
            logging_steps=50,
            eval_steps=200,
            save_steps=400,
            eval_strategy="steps",
            save_strategy="steps",
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            remove_unused_columns=False,
            dataloader_pin_memory=False,
            fp16=False,  # CPU에서는 fp16 사용 안 함
            dataloader_num_workers=0,  # CPU 최적화
            report_to=None,
            save_total_limit=1
        )
        
        # 데이터 콜레이터
        data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model,
            padding=True,
            return_tensors="pt"
        )
        
        # 트레이너 초기화
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=tokenizer,
            data_collator=data_collator
        )
        
        print("🏃‍♂️ CPU 모델 훈련 시작...")
        trainer.train()
        
        print("✅ 훈련 완료!")
        return model, tokenizer
        
    except Exception as e:
        print(f"❌ 훈련 중 에러 발생: {str(e)}")
        return None, None

# 훈련 실행
if len(papers) > 0:
    cpu_model, tokenizer = train_cpu_optimized_model(train_papers, val_papers)
    if cpu_model is not None:
        print("🎉 CPU 훈련 성공!")
    else:
        print("❌ CPU 훈련 실패")
else:
    print("❌ 데이터가 로드되지 않았습니다.")

print("✅ 모델 훈련 완료!")

🚀 CPU 최적화 Adaptive Attention T5 훈련 시작
🔧 모델 파라미터: 60,574,469
🏃‍♂️ CPU 모델 훈련 시작...
❌ 훈련 중 에러 발생: T5Stack.forward() got an unexpected keyword argument 'labels'
❌ CPU 훈련 실패
✅ 모델 훈련 완료!
